In [2]:
import pandas as pd

# Load the preprocessed dataset
df = pd.read_parquet('3_delta_replaced_data.parquet')

df.head(-1)

,timestamp,ase_spec,br_immed_spec,br_indirect_spec,br_mis_pred,br_pred,br_return_spec,branch-load-misses,branch-loads,branch-misses,...,writeback_writeback_single_inode,writeback_writeback_single_inode_start,writeback_writeback_start,writeback_writeback_wait,writeback_writeback_wait_iff_congested,writeback_writeback_wake_background,writeback_writeback_write_inode,writeback_writeback_written,attack,Label
0,2025-07-22T18:00:00,842.0,534639.0,114176.0,14852.0,648716.0,95435.0,15470.0,657359.0,16248.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,benign,benign
1,2025-07-22T18:00:10,830.0,530504.0,113038.0,14845.0,643445.0,94416.0,14958.0,649089.0,15080.0,...,2.0,2.0,12.0,0.0,0.0,0.0,2.0,12.0,benign,benign
2,2025-07-22T18:00:20,775.0,530791.0,113318.0,15235.0,644009.0,94677.0,16126.0,678009.0,14637.0,...,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,benign,benign
3,2025-07-22T18:00:30,895.0,533606.0,113935.0,14633.0,647442.0,95136.0,15474.0,646697.0,14399.0,...,0.0,0.0,-2.0,0.0,0.0,0.0,-1.0,-2.0,benign,benign
4,2025-07-22T18:00:40,822.0,536215.0,114003.0,15363.0,650120.0,95225.0,14512.0,640024.0,14608.0,...,-1.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,benign,benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6788,2025-07-23T12:51:20,865.0,536569.0,115208.0,15914.0,651677.0,96198.0,15994.0,665344.0,15742.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,benign,benign
6789,2025-07-23T12:51:30,865.0,536569.0,115208.0,15914.0,651677.0,96198.0,15994.0,665344.0,15742.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,benign,benign
6790,2025-07-23T12:51:40,865.0,536569.0,115208.0,15914.0,651677.0,96198.0,15994.0,665344.0,15742.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,benign,benign
6791,2025-07-23T12:51:50,865.0,536569.0,115208.0,15914.0,651677.0,96198.0,15994.0,665344.0,15742.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,benign,benign


In [3]:
# Split the dataset into 10 time-ordered blocks
block_size = len(df) // 10
blocks = [df.iloc[i * block_size:(i + 1) * block_size] for i in range(9)]
blocks.append(df.iloc[9 * block_size:])  # Last block takes the remaining data

In [4]:
# Print class distribution in each block
label_counts = [block['Label'].value_counts() for block in blocks]
for i, counts in enumerate(label_counts):
    print(f'Block {i + 1}:', dict(counts))

Block 1: {'benign': np.int64(679)}
Block 2: {'benign': np.int64(679)}
Block 3: {'benign': np.int64(679)}
Block 4: {'benign': np.int64(679)}
Block 5: {'benign': np.int64(679)}
Block 6: {'benign': np.int64(679)}
Block 7: {'benign': np.int64(589), 'attack': np.int64(90)}
Block 8: {'benign': np.int64(369), 'attack': np.int64(310)}
Block 9: {'benign': np.int64(372), 'attack': np.int64(307)}
Block 10: {'benign': np.int64(456), 'attack': np.int64(227)}


In [5]:
# Use blocks 2, 5, and 8 as test data, the rest as training data
test_blocks = [blocks[1], blocks[4], blocks[8]]
train_blocks = [block for i, block in enumerate(blocks) if i not in [1, 4, 8]]

In [6]:
# Concatenate the blocks to form final datasets
test_df = pd.concat(test_blocks).reset_index(drop=True)
train_df = pd.concat(train_blocks).reset_index(drop=True)

# Save the resulting datasets as Parquet files
test_df.to_parquet('test_block.parquet')
train_df.to_parquet('train_block.parquet')

# Reload saved Parquet files (for verification or further use)
train_df = pd.read_parquet('train_block.parquet')
test_df = pd.read_parquet('test_block.parquet')

In [7]:
# Function to compute label distribution percentages
def label_percentages(df):
    return (df['Label'].value_counts(normalize=True) * 100).round(2)

# Display label distribution
print("Train Set:")
print(label_percentages(train_df))

print("\nTest Set:")
print(label_percentages(test_df))

Train Set:
Label
benign    86.82
attack    13.18
Name: proportion, dtype: float64

Test Set:
Label
benign    84.93
attack    15.07
Name: proportion, dtype: float64


In [8]:
import pandas as pd
test_df = pd.read_parquet("test_block.parquet")


test_df.head(-1)

,timestamp,ase_spec,br_immed_spec,br_indirect_spec,br_mis_pred,br_pred,br_return_spec,branch-load-misses,branch-loads,branch-misses,...,writeback_writeback_single_inode,writeback_writeback_single_inode_start,writeback_writeback_start,writeback_writeback_wait,writeback_writeback_wait_iff_congested,writeback_writeback_wake_background,writeback_writeback_write_inode,writeback_writeback_written,attack,Label
0,2025-07-22T19:53:10,847.0,540525.0,115706.0,15452.0,656131.0,96680.0,15185.0,651088.0,14893.0,...,-4.0,-4.0,-6.0,0.0,0.0,0.0,-3.0,-6.0,benign,benign
1,2025-07-22T19:53:20,808.0,527069.0,112884.0,14933.0,639855.0,94312.0,14941.0,640294.0,14456.0,...,5.0,5.0,6.0,0.0,0.0,0.0,4.0,6.0,benign,benign
2,2025-07-22T19:53:30,844.0,537613.0,115032.0,15021.0,652547.0,96070.0,14841.0,642126.0,15904.0,...,1.0,1.0,-1.0,0.0,0.0,0.0,1.0,-1.0,benign,benign
3,2025-07-22T19:53:40,879.0,530950.0,113144.0,14686.0,643996.0,94525.0,14882.0,640138.0,15161.0,...,-2.0,-2.0,-2.0,0.0,0.0,0.0,-1.0,-2.0,benign,benign
4,2025-07-22T19:53:50,872.0,533494.0,113819.0,14967.0,647215.0,95060.0,14997.0,650649.0,14580.0,...,-6.0,-6.0,-2.0,0.0,0.0,0.0,-5.0,-2.0,benign,benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2031,2025-07-23T10:57:30,806.0,533503.0,113781.0,14646.0,647184.0,95055.0,14664.0,637990.0,15208.0,...,-1.0,-1.0,-2.0,0.0,0.0,0.0,-1.0,-2.0,slowloris,attack
2032,2025-07-23T10:57:40,894.0,542174.0,116248.0,15241.0,658323.0,97140.0,15403.0,648699.0,15138.0,...,-3.0,-3.0,-5.0,0.0,0.0,0.0,-3.0,-5.0,slowloris,attack
2033,2025-07-23T10:57:50,807.0,539645.0,114957.0,15198.0,654503.0,95993.0,14782.0,649298.0,15206.0,...,3.0,3.0,2.0,0.0,0.0,0.0,2.0,2.0,slowloris,attack
2034,2025-07-23T10:58:00,848.0,538522.0,114847.0,15208.0,653269.0,95939.0,15291.0,647818.0,16192.0,...,1.0,1.0,2.0,0.0,0.0,0.0,2.0,2.0,slowloris,attack
